# Daily Challenge — BaristaBot with LangGraph and Gemini API
**Developers Institute | Full Stack & AI Bootcamp**

Complete notebook following the challenge's progression: single-turn graph → conversational loop → live menu tool → stateful ordering system. Designed for **Google Colab**. The menu and order-tool signatures match the assignment. The Gemini model is set to `gemini-3.5-flash-lite`, which was verified working during development; availability depends on your API account.

> **Important:** In Colab, add `GOOGLE_API_KEY` under **Secrets** (🔑) and grant this notebook access. No API key is embedded in this file.

## 1 — Install dependencies

In [ ]:
%pip install -q "langgraph==1.0.5" "langgraph-prebuilt==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"

## 2 — Configure the Gemini API key (Colab Secrets)

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Add GOOGLE_API_KEY to Colab Secrets and grant access.")
os.environ["GOOGLE_API_KEY"] = api_key
print("Gemini API key loaded from Colab Secrets.")

## 3 — Define `OrderState` and system instructions
The graph stores the conversation history, a list of ordered drinks with modifiers, and whether the order has been placed.

In [ ]:
from typing import Annotated, TypedDict, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

class OrderState(TypedDict):
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool
    confirmed_order: tuple[str, ...] | None

BARISTABOT_SYSINT = """You are BaristaBot, an interactive cafe ordering assistant.
Only discuss menu items and cafe orders. Be friendly, brief, and accurate.
Use get_menu to find the actual available drinks and modifiers. Never invent menu items.
Ask clarifying questions for unavailable or ambiguous products or modifiers.
Add drinks with add_to_order(drink, modifiers); modifiers must be a list of strings.
Use get_order to check the structured order; use clear_order to restart an order.
When the customer indicates they are done, call confirm_order. This tool prints the
EXACT structured order and asks the customer directly for confirmation or changes.
If the customer requests changes, apply them and call confirm_order again.
Call place_order ONLY after confirm_order has returned an explicit affirmative response
for the current order. Do not place an empty order.
The cafe may be out of stock of soy milk; follow get_menu exactly.
You may request menu and ordering tools together when appropriate.
When the order has been placed, thank the customer and say goodbye.
"""
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type q to quit. How may I serve you today?"


## 4 — Single-turn chatbot graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)

def single_turn_chatbot(state: OrderState):
    response = llm.invoke([SystemMessage(content=BARISTABOT_SYSINT)] + state["messages"])
    return {"messages": [response]}

single_builder = StateGraph(OrderState)
single_builder.add_node("chatbot", single_turn_chatbot)
single_builder.add_edge(START, "chatbot")
single_builder.add_edge("chatbot", END)
chat_graph = single_builder.compile()
print("Single-turn graph compiled.")

## 5 — Visualize and test the single-turn graph

In [ ]:
from IPython.display import Image, display
try:
    display(Image(chat_graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Graph image unavailable:", exc)
    print(chat_graph.get_graph().draw_mermaid())

In [ ]:
state = chat_graph.invoke({
    "messages": [HumanMessage(content="Hello! What do you do?")],
    "order": [], "finished": False,
})
for message in state["messages"]:
    print(f"{type(message).__name__}: {message.content}")

## 6 — A second conversational turn

In [ ]:
state = chat_graph.invoke({
    **state,
    "messages": state["messages"] + [HumanMessage(content="Can you help me order a coffee?")],
})
for message in state["messages"][-2:]:
    print(f"{type(message).__name__}: {message.content}")

## 7 — Human node and conditional conversational loop
The interactive graph prints assistant responses and ends when the user types `q`, `quit`, `exit`, or `goodbye`. This is a *demonstration*; the full ordering graph is built later.

In [ ]:
def render_content(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(
            part.get("text", "") for part in content
            if isinstance(part, dict) and part.get("type") == "text"
        )
    return str(content)

def human_node(state: OrderState):
    """Print the last AI message and ask for input."""
    last_message = state["messages"][-1]
    answer = render_content(last_message.content)
    if answer:
        print("\nBaristaBot:", answer, "\n")
    user_input = input("You: ").strip()
    if user_input.lower() in {"q", "quit", "exit", "goodbye"}:
        return {"messages": [HumanMessage(content=user_input)], "finished": True}
    return {"messages": [HumanMessage(content=user_input)]}

def chatbot_with_welcome_msg(state: OrderState):
    if not state["messages"]:
        return {"messages": [AIMessage(content=WELCOME_MSG)]}
    response = llm.invoke([SystemMessage(content=BARISTABOT_SYSINT)] + state["messages"])
    return {"messages": [response]}

def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    return END if state.get("finished", False) else "chatbot"

human_builder = StateGraph(OrderState)
human_builder.add_node("chatbot", chatbot_with_welcome_msg)
human_builder.add_node("human", human_node)
human_builder.add_edge(START, "chatbot")
human_builder.add_edge("chatbot", "human")
human_builder.add_conditional_edges("human", maybe_exit_human_node)
chat_with_human_graph = human_builder.compile()
try:
    display(Image(chat_with_human_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(chat_with_human_graph.get_graph().draw_mermaid())
print("Human-loop graph ready. Optional test: chat_with_human_graph.invoke({'messages': [], 'order': [], 'finished': False}, {'recursion_limit': 100})")

## 8 — Live menu (`get_menu`)
This is the **exact menu and modifier list from the assignment**, including the unavailable soy milk. The exercise does not supply prices or food items, so none are invented.

In [ ]:
from langchain_core.tools import tool

@tool
def get_menu() -> str:
    """Provide the latest up-to-date cafe menu and modifier availability."""
    return """MENU:
Coffee Drinks:
  Espresso
  Americano
  Cold Brew
Coffee Drinks with Milk:
  Latte
  Cappuccino
  Cortado
  Macchiato
  Mocha
  Flat White
Tea Drinks:
  English Breakfast Tea
  Green Tea
  Earl Grey
Tea Drinks with Milk:
  Chai Latte
  Matcha Latte
  London Fog
Other Drinks:
  Steamer
  Hot Chocolate
Modifiers:
  Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
  Espresso shots: Single, Double, Triple, Quadruple; default: Double
  Caffeine: Decaf, Regular; default: Regular
  Hot-Iced: Hot, Iced; Default: Hot
  Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
  Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.
  'dirty' means add a shot of espresso to a drink that doesn't usually have it, like 'Dirty Chai Latte'.
  'Regular milk' is the same as 'whole milk'.
  'Sweetened' means add some regular sugar, not a sweetener.
  Soy milk has run out of stock today, so soy is not available.
"""
print(get_menu.invoke({}))

## 9 — Menu-only `ToolNode` demonstration

In [ ]:
from langgraph.prebuilt import ToolNode

auto_tools = [get_menu]
menu_tool_node = ToolNode(auto_tools)
llm_with_menu = llm.bind_tools(auto_tools)

def chatbot_with_menu(state: OrderState):
    if not state["messages"]:
        return {"messages": [AIMessage(content=WELCOME_MSG)]}
    return {"messages": [llm_with_menu.invoke(
        [SystemMessage(content=BARISTABOT_SYSINT)] + state["messages"]
    )]}

def route_menu(state: OrderState) -> Literal["tools", "human"]:
    return "tools" if getattr(state["messages"][-1], "tool_calls", []) else "human"

menu_builder = StateGraph(OrderState)
menu_builder.add_node("chatbot", chatbot_with_menu)
menu_builder.add_node("human", human_node)
menu_builder.add_node("tools", menu_tool_node)
menu_builder.add_edge(START, "chatbot")
menu_builder.add_conditional_edges("chatbot", route_menu)
menu_builder.add_conditional_edges("human", maybe_exit_human_node)
menu_builder.add_edge("tools", "chatbot")
graph_with_menu = menu_builder.compile()
try:
    display(Image(graph_with_menu.get_graph().draw_mermaid_png()))
except Exception:
    print(graph_with_menu.get_graph().draw_mermaid())

## 10 — Order-tool schemas and the stateful `ordering` node
The menu is handled automatically by `ToolNode`. The five order tools are schemas exposed to Gemini; the **separate `ordering` node** implements their state changes, asks for confirmation, and supplies a kitchen ETA.

In [ ]:
from collections.abc import Iterable
from random import randint

@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Add a menu drink and its modifiers to the customer's order."""
    raise NotImplementedError("Handled by the ordering graph node")

@tool
def confirm_order() -> str:
    """Display the exact order and ask the customer to confirm or request changes."""
    raise NotImplementedError("Handled by the ordering graph node")

@tool
def get_order() -> str:
    """Return the current structured order, one item per line."""
    raise NotImplementedError("Handled by the ordering graph node")

@tool
def clear_order():
    """Remove all items from the current order."""
    raise NotImplementedError("Handled by the ordering graph node")

@tool
def place_order() -> int:
    """Send the confirmed order to the kitchen and return an ETA in minutes."""
    raise NotImplementedError("Handled by the ordering graph node")

order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

def ordering_node(state: OrderState):
    """Handle order tools, updating order and conversation messages."""
    msg = state["messages"][-1]
    order = list(state.get("order", []))
    results = []
    placed = False
    confirmed_snapshot = state.get("confirmed_order")
    for call in msg.tool_calls:
        name = call["name"]
        args = call.get("args", {})
        if name == "add_to_order":
            drink = args["drink"]
            modifiers = args.get("modifiers", []) or []
            if isinstance(modifiers, str):
                modifiers = [modifiers]
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"
            order.append(f"{drink} ({modifier_str})")
            confirmed_snapshot = None
            response = "\n".join(order)
        elif name == "get_order":
            response = "\n".join(order) if order else "(no order)"
        elif name == "clear_order":
            order.clear()
            confirmed_snapshot = None
            response = "Order cleared."
        elif name == "confirm_order":
            print("\nYour order:")
            for drink in order:
                print(" -", drink)
            if not order:
                print(" (no items)")
                response = "The order is empty; ask the customer to select a drink."
            else:
                response = input("Is this correct? (yes / describe changes): ").strip()
                if response.lower() in {"yes", "y", "sim", "sí", "si", "correct", "confirmed", "confirm"}:
                    confirmed_snapshot = tuple(order)
                    response = "Customer explicitly confirmed the exact current order."
                else:
                    confirmed_snapshot = None
                    response = "Customer did not confirm. Their response: " + response
        elif name == "place_order":
            if not order:
                response = "Cannot place an empty order."
            elif confirmed_snapshot != tuple(order):
                response = "Cannot place order: request confirm_order and obtain explicit customer approval first."
            else:
                print("\nSending order to kitchen!")
                for drink in order:
                    print(" -", drink)
                placed = True
                eta = randint(1, 5)
                response = f"Order sent to kitchen. Estimated preparation time: {eta} minutes."
                print(response)
        else:
            raise NotImplementedError(f"Unknown order tool: {name}")
        results.append(ToolMessage(content=str(response), name=name, tool_call_id=call["id"]))
    return {"messages": results, "order": order, "finished": placed, "confirmed_order": confirmed_snapshot}


## 11 — Complete graph: chatbot → menu tools / ordering / human / END

In [ ]:
llm_with_tools = llm.bind_tools(auto_tools + order_tools)

def full_chatbot(state: OrderState):
    if state.get("finished", False):
        # Use the model without tools for a natural closing message.
        try:
            farewell = llm.invoke([
                SystemMessage(content="The cafe order has been placed successfully. Thank the customer and say goodbye briefly. Do not claim additional actions."),
                *state["messages"],
            ])
            return {"messages": [farewell]}
        except Exception as exc:
            print("Farewell generation unavailable:", type(exc).__name__)
            return {"messages": [AIMessage(content="Thank you! Your order has been placed. Goodbye!")]}
    if not state["messages"]:
        return {"messages": [AIMessage(content=WELCOME_MSG)]}
    try:
        reply = llm_with_tools.invoke([SystemMessage(content=BARISTABOT_SYSINT)] + state["messages"])
    except Exception as exc:
        # Avoid an unhandled API exception during interactive demonstrations.
        print(f"Gemini request failed ({type(exc).__name__}): {exc}")
        reply = AIMessage(content="Sorry, I couldn't contact the ordering assistant. Please try again, or type q to quit.")
    return {"messages": [reply]}

def route_full_chatbot(state: OrderState) -> Literal["tools", "ordering", "mixed", "human", "__end__"]:
    if state.get("finished", False):
        print("\nBaristaBot:", render_content(state["messages"][-1].content))
        return END
    calls = getattr(state["messages"][-1], "tool_calls", [])
    if not calls:
        return "human"
    menu_calls = [c for c in calls if c["name"] == "get_menu"]
    order_calls = [c for c in calls if c["name"] != "get_menu"]
    if menu_calls and order_calls:
        return "mixed"
    return "tools" if menu_calls else "ordering"

def mixed_tools_node(state: OrderState):
    """Answer every tool call in a mixed batch without losing tool_call IDs."""
    original = state["messages"][-1]
    menu_calls = [c for c in original.tool_calls if c["name"] == "get_menu"]
    order_calls = [c for c in original.tool_calls if c["name"] != "get_menu"]
    menu_messages = [ToolMessage(content=get_menu.invoke({}), name="get_menu", tool_call_id=c["id"])
                     for c in menu_calls]
    if not order_calls:
        return {"messages": menu_messages}
    # Preserve all order tool IDs while invoking the existing ordering node.
    filtered = AIMessage(content="", tool_calls=order_calls)
    result = ordering_node({**state, "messages": [*state["messages"][:-1], filtered]})
    return {**result, "messages": menu_messages + result["messages"]}

full_builder = StateGraph(OrderState)
full_builder.add_node("chatbot", full_chatbot)
full_builder.add_node("human", human_node)
full_builder.add_node("tools", menu_tool_node)
full_builder.add_node("ordering", ordering_node)
full_builder.add_node("mixed", mixed_tools_node)
full_builder.add_edge(START, "chatbot")
full_builder.add_conditional_edges("chatbot", route_full_chatbot)
full_builder.add_conditional_edges("human", maybe_exit_human_node)
full_builder.add_edge("tools", "chatbot")
full_builder.add_edge("ordering", "chatbot")
full_builder.add_edge("mixed", "chatbot")
graph_with_order_tools = full_builder.compile()
try:
    display(Image(graph_with_order_tools.get_graph().draw_mermaid_png()))
except Exception:
    print(graph_with_order_tools.get_graph().draw_mermaid())
print("Full graph compiled: chatbot, human, tools, ordering, mixed.")


## 12 — Run the complete BaristaBot
**Suggested test:** Ask for the menu; order a *hot oat-milk cappuccino*; ask to review the order; request confirmation; answer **yes** at the `Is this correct?` prompt. The bot should send the order to the kitchen, show an ETA, thank you, and exit. Test `clear_order` and order changes in a separate run.

> This last cell is interactive. Run it only after the preceding cells have completed.

In [ ]:
config = {"recursion_limit": 100}
final_state = graph_with_order_tools.invoke(
    {"messages": [], "order": [], "finished": False, "confirmed_order": None}, config=config
)
print("\nFinal order:", final_state["order"])
print("Conversation finished:", final_state["finished"])
print("Tools executed:")
for message in final_state["messages"]:
    if isinstance(message, AIMessage):
        for call in getattr(message, "tool_calls", []):
            print(" -", call["name"], call.get("args", {}))

## Submission checklist
- [ ] Run all cells in Google Colab with `GOOGLE_API_KEY` configured.
- [ ] Verify the single-turn graph and human-loop graph render.
- [ ] Verify the original assignment menu is displayed and soy milk is unavailable.
- [ ] Test adding drinks with modifiers, order review, clearing, changes, confirmation, and placement.
- [ ] Confirm that the kitchen ETA and final goodbye are displayed.
- [ ] Save the executed notebook and push the `.ipynb` file to GitHub.

**Implementation note:** This is an educational in-memory demonstration. The order-confirmation snapshot is a notebook-level variable reset before each run; a production multi-user system should keep that confirmation state per session (e.g., in graph state/checkpoints).